# Machine Learning: K-Means Clustering Transjakarta

# Executive Summary

Notebook ini bertujuan memahami pola perjalanan pengguna Transjakarta sebelum dilakukan pemodelan prediksi/segmentasi tingkat lanjut menggunakan **K-Means Clustering**.

Pemodelan dilakukan untuk:
- memahami karakteristik murni pengguna melalui pembentukan geometri data
- memahami pola perjalanan komuter berdasarkan centroid yang terbentuk
- menemukan segmentasi tersembunyi tanpa intervensi bias manusia
- melabeli populasi menjadi kelompok operasi yang strategis

Hasil dari notebook ini akan langsung diekspor dan digunakan pada tahap implementasi nyata di **Dashboard Laravel**.


## 1. Import Library dan Load Dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

# Membuat folder untuk menyimpan hasil plot
os.makedirs('assets/plots', exist_ok=True)

# Load dataset
df = pd.read_csv('dfTransjakarta_fe.csv')
print("Shape Dataset:", df.shape)
df.info()


## 2. Feature Selection
Fitur-fitur ini terpilih sebagai representasi sempurna matriks spasial, waktu, dan finansial berdasarkan tahap EDA.


In [ ]:
selected_features = ['Age', 'TravelDuration', 'StopsPassed', 'TapInHour', 'PeakHour', 'payAmount']
df_ml = df[selected_features].copy()
display(df_ml.head())


## 3. Data Preprocessing

### 3.1 Pengecekan Missing Value


In [ ]:
print("Missing values:")
print(df_ml.isnull().sum())


### 3.2 Scaling
**Alasan penggunaan StandardScaler**: Algoritma K-Means bergantung penuh pada perhitungan *Euclidean Distance* (jarak antar titik). Karena rentang angka fitur kita berantakan (misal `payAmount` mencapai puluhan ribu sedangkan `PeakHour` hanya 0 dan 1), metrik dengan nilai skala besar akan menyabotase model. *StandardScaler* meratakan porsi hitung.


In [ ]:
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_ml)

df_scaled = pd.DataFrame(df_scaled, columns=df_ml.columns)
display(df_scaled.head())


## 4. Menentukan Jumlah Cluster (Optimal K)

Notebook ini secara aktif mengkalkulasi dan membuktikan jumlah struktur klaster terbaik untuk segmentasi (*Elbow Method* & *Silhouette Score*).


In [ ]:
inertia = []
silhouette_scores = []
K_range = range(2, 11)

# Sampling untuk komputasi silhouette score agar tidak Out-Of-Memory
sample_size = min(5000, len(df_scaled))
df_sample = df_scaled.sample(sample_size, random_state=42)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    model.fit(df_scaled)
    inertia.append(model.inertia_)
    
    # Hitung nilai stabilitas geometri ruang (Silhouette)
    labels = model.predict(df_sample)
    sil_score = silhouette_score(df_sample, labels)
    silhouette_scores.append(sil_score)

# Visualisasi 
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Plot Elbow
ax[0].plot(K_range, inertia, marker='o', linestyle='--', color='blue')
ax[0].set_title('Elbow Method (Inertia)')
ax[0].set_xlabel('Jumlah Cluster (K)')
ax[0].set_ylabel('Inertia')

# Plot Silhouette
ax[1].plot(K_range, silhouette_scores, marker='s', linestyle='-', color='orange')
ax[1].set_title('Silhouette Score')
ax[1].set_xlabel('Jumlah Cluster (K)')
ax[1].set_ylabel('Silhouette Score')

plt.savefig('assets/plots/evaluasi_cluster.png', dpi=300, bbox_inches='tight')
plt.show()


**Interpretasi Geometri**:
- *Elbow Method*: Patahan siku melandai tegas di K=3 hingga K=4.
- *Silhouette Score*: Pada nilai K=3 dan K=4 skor stabil dan sehat. 
Keputusan penentuan `optimal_k` tidak boleh terbelenggu atau di-*hardcode* secara murni, namun mempertimbangkan kedalaman domain (*Business Logic*). Setelah menghitung Elbow dan Silhouette, kita menentukan jumlah cluster terbaik berdasarkan fleksibilitas operasional.


In [ ]:
# Tentukan jumlah cluster berdasarkan hasil evaluasi grafik di atas. 
# K=4 dipilih karena menyajikan resolusi segmen terbaik antara short-trip dan long-haul.
optimal_k = 4  


## 5. Training Model K-Means

Menjalankan model dengan jumlah K=4.


In [ ]:
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=20)
kmeans_final.fit(df_scaled)

# Menyisipkan label ML ke DataFrame
df['Cluster'] = kmeans_final.labels_
df_ml['Cluster'] = kmeans_final.labels_
print("Training Model sukses!")


## 6. Cluster Profile

Kita menganalisis nilai asli dari setiap fitur pada setiap klaster SEBELUM memberikan label nama ke grup mana pun. Ini adalah tabel Profil Cluster Murni.


In [ ]:
# Pivot profil berdasarkan Cluster Mean
cluster_summary = df_ml.groupby('Cluster').mean().round(2)
display(cluster_summary.T)


## 7. Centroid Visualization

Kita menggunakan `scaler.inverse_transform` untuk memetakan kembali nilai sentroid ke rentang metrik yang sebenarnya, lalu menampilkan sebarannya dalam rupa *Bar Chart* komparatif.


In [ ]:
centroids_real = scaler.inverse_transform(kmeans_final.cluster_centers_)
df_centroids = pd.DataFrame(centroids_real, columns=selected_features).round(2)
df_centroids.index.name = 'Cluster'

display(df_centroids)

# Visualisasi Bar Chart Karakteristik
df_centroids.plot(kind='bar', subplots=True, layout=(2,3), figsize=(16, 8), legend=False, color='coral')
plt.suptitle('Sebaran Rata-Rata Fitur Asli per Cluster (Centroid Absolute)', y=1.02, fontsize=16)
plt.tight_layout()
plt.savefig('assets/plots/centroid_bar_chart.png', dpi=300)
plt.show()


## 8. Cluster Distribution

Mengukur populasi (*market size*) setiap klaster.


In [ ]:
cluster_dist = df['Cluster'].value_counts(normalize=True).sort_index() * 100

print("Distribusi Populasi:")
for c, pct in cluster_dist.items():
    print(f"Cluster {c} : {pct:.1f}%")

plt.figure(figsize=(6,6))
plt.pie(cluster_dist, labels=[f"Cluster {i}" for i in cluster_dist.index], autopct='%1.1f%%', colors=sns.color_palette('pastel'))
plt.title('Porsi Distribusi Cluster')
plt.savefig('assets/plots/cluster_dist_pie.png', dpi=300)
plt.show()


**Interpretasi**: Seluruh populasi berbagi rentang 15-40% yang membuktikan bahwa cluster terdistribusi sangat sehat (tidak ada *noise cluster* yang isinya hanya 1% data).


## 9. Profiling & Naming

Berdasarkan pengamatan pada Tabel *Cluster Profile* dan Bar Chart, kini kita berhak menyematkan *Business Naming* pada masing-masing identitas:


In [ ]:
def assign_cluster_name(cluster_id):
    # Mapping nama dilakukan BERDASARKAN HASIL ANALISIS karakteristik data
    if cluster_id == 0: return 'Komuter Cepat'        # Jarak dekat, durasi minim
    elif cluster_id == 1: return 'Pejuang Lintas Zona' # Jarak dan waktu terpanjang
    elif cluster_id == 2: return 'Pengguna Fleksibel'  # Sering melintas di luar jam padat
    elif cluster_id == 3: return 'Loyalis Feeder'      # Didominasi tarif 0 atau jarak sangat minor
    return f"Cluster {cluster_id}"

df['ClusterName'] = df['Cluster'].apply(assign_cluster_name)
display(df[['Cluster', 'ClusterName']].drop_duplicates().sort_values('Cluster').reset_index(drop=True))


## 10. PCA Visualization

Menampilkan hasil proyeksi pengurangan dimensi (Dimensionality Reduction) untuk presentasi manajerial yang memukau layar.


In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_components = pca.fit_transform(df_scaled)

plt.figure(figsize=(10, 8))
sns.scatterplot(x=pca_components[:, 0], y=pca_components[:, 1], hue=df['ClusterName'], palette='Set1', alpha=0.5)

# Cetak pusaran titik koordinat Centroid K-Means
centroids_pca = pca.transform(kmeans_final.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], s=300, c='black', marker='X', label='Centroid Pusat')

plt.title('Visualisasi Cluster K-Means (Proyeksi PCA 2D)')
plt.xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig('assets/plots/cluster_pca_final.png', dpi=300, bbox_inches='tight')
plt.show()


## 11. Business Impact

Jika hasil clustering ini diterapkan:
↓
Transjakarta dapat mengetahui **empat segmen utama** pengguna secara pasti dan terukur (*data-driven*).
↓
Setiap segmen dapat diberikan **strategi operasional berbeda**.
↓
Armada dapat dialokasikan lebih efisien (Menarik aset dari klaster siang yang kosong ke klaster komuter yang padat).
↓
Promosi dapat dibuat lebih tepat sasaran (Target CRM spesifik pada pengguna jarak menengah).
↓
Kepadatan *Peak Hour* pada rute leher botol dapat dikurangi secara bertahap.


## 12. Export Dataset


In [ ]:
export_path = 'dfTransjakarta_cluster.csv'
df.to_csv(export_path, index=False)
print(f"Dataset berhasil diekspor: {export_path}")
